# Banking Customer Analytics - EDA
End-to-end exploratory data analysis on 3,000 banking customers.

> **Setup:** Create a `.env` file in the project root with your DB credentials before running. See README.md.

In [ ]:
# --- DB Connection (reads credentials from .env file) ---
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from project root

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'banking_case')

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:3306/{DB_NAME}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_sql("SELECT * FROM customer", engine)

## 1. Data Overview

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Generate descriptive statistics for the dataframe
df.describe()

## 2. Income Band Segmentation

In [ ]:
bins = [0, 100000, 300000, float('inf')]
labels = ["Low", "Medium", "High"]

df["Income Band"] = pd.cut(df["Estimated Income"], bins=bins, labels=labels, right=False)

df["Income Band"].value_counts().plot(kind="bar")

## 3. Categorical Analysis

In [ ]:
Categorical_cols = df[[
    "BRId",
    "GenderId",
    "IAId",
    "Amount of Credit Cards",
    "Nationality",
    "Occupation",
    "Fee Structure",
    "Loyalty Classification"
]].columns

for col in Categorical_cols:
    print(f"\nValue counts for '{col}':")
    display(df[col].value_counts())

## 4. Univariate Analysis

In [ ]:
for i, predictor in enumerate(df[["BRId", "GenderId", "IAId", "Amount of Credit Cards",
                                 "Nationality", "Occupation", "Fee Structure",
                                 "Loyalty Classification"]]):
    plt.figure(i)
    sns.countplot(data=df, x=predictor)

## 5. Bivariate Analysis

In [ ]:
for i, predictor in enumerate(df[["BRId", "GenderId", "IAId", "Amount of Credit Cards",
                                 "Nationality", "Occupation", "Fee Structure",
                                 "Loyalty Classification"]]):
    plt.figure(i)
    sns.countplot(data=df, x=predictor, hue="GenderId")

In [ ]:
# Histplot of value counts for different columns
for col in Categorical_cols:
    if col == "Occupation":
        continue
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col])
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Counts")
    plt.show()

## 6. Numerical Analysis

In [ ]:
numerical_cols = ['Estimated Income', 'Superannuation Savings',
                  'Credit Card Balance', 'Bank Loans', 'Foreign Currency Account', 'Business Lending']

plt.figure(figsize=(15, 10))

for i, col in enumerate(numerical_cols):
    plt.subplot(4, 3, i+1)
    sns.histplot(df[col], kde=True)
    plt.title(col)

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
# Heatmap - Correlation between all numerical columns
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(12, 12))
sns.heatmap(correlation_matrix, annot=True, cmap='crest', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## 8. Key Insights

1. **Strongest positive correlations** occur among *Bank Deposits* with *Checking Accounts*, *Savings Accounts*, and *Foreign Currency Accounts* — customers who maintain high balances in one account type tend to hold substantial funds across all account types.

2. **Income Band distribution** shows the majority of customers fall in the Medium income band (100K–300K), consistent with the Power BI dashboard showing Medium at ~53% of loans.

3. **Business Lending** (2,600M) is the largest loan component, suggesting the bank has a strong commercial/business client base.